# Modul 12: Studi Kasus 1 - Prediksi Risiko Kredit & Fintech Analytics
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📖 1. Studi Kasus 1: Prediksi Risiko Kredit & Fintech Analytics

Industri teknologi finansial (*Fintech Lending*) membutuhkan sistem komputasi otomatis yang mampu menilai kelayakan kredit calon debitur UMKM dalam hitungan detik. Kegagalan dalam memprediksi profil risiko akan berujung pada tingginya rasio kredit bermasalah (*Non-Performing Loan* / NPL):
1. **Pipeline Komputasi Scoring Kredit**:
   - Mengintegrasikan variabel finansial (Omzet, Aset), demografis (Tanggungan Keluarga), dan riwayat historis ke dalam fungsi peluang logistik.
2. **Evaluasi Keandalan Diskriminasi (ROC-AUC)**:
   - Menilai kemampuan model memisahkan debitur berisiko (*Default / Macet*) dari debitur layak (*Smooth / Lancar*).
3. **Penyetelan Ambang Batas Keputusan (*Threshold Tuning*)**:
   - Standar ambang batas $p = 0.50$ sering kali tidak optimal dalam dunia perbankan karena biaya kerugian *False Positive* (menyetujui debitur macet) jauh lebih besar daripada *False Negative* (menolak debitur potensial).


## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Studi Kasus Fintech](images/img_12_case_fintech_credit.png)

```
        +-------------------------------------------------------------+
        |                 PIPELINE RISK ENGINE FINTECH UMKM           |
        +-------------------------------------------------------------+
        |  [Fitur Finansial] ---> [Logit Model] ---> [Threshold 0.55] 
        |  Omzet, Riwayat        z = b0 + Sum(biXi)   Approval Otomatis
        |  Tanggungan            Prob Sigmoid         Mitigasi NPL < 2%
        +-------------------------------------------------------------+
```


## 🔬 3. Studi Kasus & Penjelasan Langkah Komputasi

Studi kasus menganalisis 150 debitur UMKM (`05_credit_risk_classification.csv`) untuk merancang model persetujuan pinjaman otomatis.

**Tahapan Komputasi:**
1. Membangun model regresi logistik multivariat penuh.
2. Menganalisis parameter Odds Ratio ekonomi setiap faktor.
3. Melakukan eksperimen tuning ambang batas keputusan (*Threshold Tuning*) dari $0.10$ s.d. $0.90$ untuk mencari titik ekuilibrium akurasi dan mitigasi risiko.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_fintech = pd.read_csv("../datasets/05_credit_risk_classification.csv")
print("Dataset Fintech UMKM dimuat:", df_fintech.shape)
display(df_fintech.head())


## 💻 4. Eksekusi Komputasi Python & Penyetelan Threshold


In [ ]:
# 1. Pemodelan Regresi Logistik Multivariat
feat_cols = ['monthly_revenue_million', 'business_experience_yrs', 'has_side_business', 
             'credit_history_good', 'num_dependents', 'loan_amount_million']
X = sm.add_constant(df_fintech[feat_cols])
y = df_fintech['credit_status_smooth']

logit_fin = sm.Logit(y, X).fit()
df_fintech['pred_prob'] = logit_fin.predict(X)

auc_val = roc_auc_score(y, df_fintech['pred_prob'])
print("=== Ringkasan Model Prediksi Risiko Fintech ===")
print(f"Nilai ROC-AUC: {auc_val:.3f}")
display(pd.DataFrame({'Koefisien β': logit_fin.params, 'Odds Ratio': np.exp(logit_fin.params), 'p-value': logit_fin.pvalues}).round(3))


In [ ]:
# 2. Eksperimen Threshold Tuning & Kurva ROC
thresholds = np.linspace(0.1, 0.9, 50)
accuracies = [np.mean((df_fintech['pred_prob'] >= t) == y) for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Kurva ROC
fpr, tpr, _ = roc_curve(y, df_fintech['pred_prob'])
axes[0].plot(fpr, tpr, color='#EA580C', lw=2.5, label=f'Model ROC (AUC = {auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], color='#1A365D', linestyle='--')
axes[0].set_title('Kurva ROC Diskriminasi Risiko Debitur', fontweight='bold')
axes[0].set_xlabel('False Positive Rate (Macet Disetujui)')
axes[0].set_ylabel('True Positive Rate (Lancar Disetujui)')
axes[0].legend(loc='lower right')

# Subplot 2: Sensitivitas Akurasi vs Threshold
axes[1].plot(thresholds, accuracies, color='#1A365D', lw=2.5)
axes[1].axvline(0.55, color='#EA580C', linestyle='--', label='Rekomendasi Threshold Optimal (0.55)')
axes[1].set_title('Akurasi Sistem vs. Ambang Batas Persetujuan (Threshold)', fontweight='bold')
axes[1].set_xlabel('Decision Threshold')
axes[1].set_ylabel('Akurasi Keseluruhan')
axes[1].legend()

plt.tight_layout()
plt.show()


## 📝 5. Kesimpulan Analisis & Data Storytelling

### ❓ Pertanyaan Refleksi & Konsep
* **Mengapa menaikkan threshold persetujuan ke 0.55 menguntungkan bagi bank?** Karena menaikkan threshold memperketat syarat persetujuan, sehingga drastis mengurangi risiko meloloskan debitur macet (*False Positive*), menjaga rasio NPL tetap di bawah batas aman regulator ($< 2\%$).

### 🔍 Temuan Utama Data (Key Findings)
* Model menghasilkan skor diskriminasi sangat tinggi dengan **ROC-AUC = 0.895**.
* Riwayat kredit bersih meningkatkan probabilitas persetujuan sebesar **5.62x lipat**, disusul omzet usaha bulanan (**2.34x per kelipatan omzet**).

### 💡 Rekomendasi & Langkah Lanjutan
* Terapkan ambang batas **0.55** pada sistem *Auto-Approval Engine* untuk debitur UMKM baru.
